# Infer cell-type-specific key regulators

The ``predict_cell_type_master_regulators`` command fine-tunes ChromBERT on cell-type-specific accessibility data (BigWig + peaks) and then infers a cell-type-specific key regulators. If a fine-tuned checkpoint is provided, fine-tuning is skipped and the key regulators is inferred directly from the checkpoint.

**Note**: The remaining examples will only show the direct command usage. 

If you need to use Singularity container, please refer to the [`singularity_use.ipynb`](singularity_use.ipynb) tutorial for detailed instructions on using `singularity exec` with `chrombert-tools`.

## Example: 
infer cell-type-specific key regulators for myoblast

In [1]:
import pandas as pd
import numpy as np
import os


In [2]:
os.environ["CUDA_VISIBLE_DEVICES"]='3'

In [3]:
!chrombert-tools -h

Usage: chrombert-tools [OPTIONS] COMMAND [ARGS]...

  Type -h or --help after any subcommand for more information.

Options:
  -v, --verbose  Verbose logging
  -d, --debug    Post mortem debugging
  -V, --version  Show the version and exit.
  -h, --help     Show this message and exit.

Commands:
  embed_region                    Generate region...
  embed_regulator                 Extract regulator...
  gene_activity_repression        Predict gene...
  interpret_region_region_interactions
                                  Region embedding...
  interpret_regulator_effects_between_region_groups
                                  Identify regulators...
  interpret_regulator_regulator_interactions
                                  Interpret...
  predict_cell_type_master_regulators
                                  Find candidate key...
  predict_regulator_context_cofactors
                                  Identify...
  predict_tf_binding_regions      Predict TF binding...
  predict_transit

In [5]:
!chrombert-tools predict_cell_type_master_regulators -h

Usage: chrombert-tools predict_cell_type_master_regulators 
           [OPTIONS]

  Find candidate key regulators for a target cell type or between two region
  sets.

  Typical usage

  Provide:     --cell-type-bw     --cell-type-peak

      a) If --ft-ckpt is provided,     The workflow will use the checkpoint
      together with the cell-type accessibility data to identify region groups
      automatically and then rank candidate key regulators.

      b) If --ft-ckpt is not provided,     The workflow will build a cell-
      specific model, define region groups from the cell-type accessibility
      data and then rank candidate key regulators.

Options:
  --cell-type-bw FILE             BigWig file of chromatin accessibility for
                                  the target cell type. Use this together with
                                  --cell-type-peak when you do not already
                                  have a fine-tuned checkpoint.
  --cell-type-peak FILE           BED fi

#### download myoblast bigwig and peak file from encode

In [6]:
# download myoblast 
import subprocess
if not os.path.exists('../data/myoblast_ENCFF647RNC_peak.bed'):
    cmd = f'wget https://www.encodeproject.org/files/ENCFF647RNC/@@download/ENCFF647RNC.bed.gz -O ../data/myoblast_ENCFF647RNC_peak.bed.gz'
    subprocess.run(cmd, shell=True)
    cmd = f"gzip -d ../data/myoblast_ENCFF647RNC_peak.bed.gz"
    subprocess.run(cmd, shell=True)

In [7]:
# import subprocess
if not os.path.exists('../data/myoblast_ENCFF149ERN_signal.bigwig'):
    cmd = f'wget https://www.encodeproject.org/files/ENCFF149ERN/@@download/ENCFF149ERN.bigWig -O ../data/myoblast_ENCFF149ERN_signal.bigwig'
    subprocess.run(cmd, shell=True)    

## Run

In [8]:
!mkdir -p ./tmp

In [9]:
# takes approximately 20-60 minutes to run
!chrombert-tools predict_cell_type_master_regulators \
    --cell-type-bw "../data/myoblast_ENCFF149ERN_signal.bigwig" \
    --cell-type-peak "../data/myoblast_ENCFF647RNC_peak.bed" \
    --odir "./output_infer_cell_key_regulator" \
    --genome "hg38" \
    --resolution "1kb"  2> "./tmp/infer_cell_key_regulator.stderr.log" # redirect stderr to log file
    

Step 1/3: Building or loading a cell-specific model...
Preparing dataset ...
Region summary - total: 373422, overlapping with ChromBERT: 368260 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 7920
Total regions: 324690
Fast mode: downsampling to 20k regions
Fine-tuning cell-specific model...

[Attempt 0/2] seed=55
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Epoch 0:  20%|████▍                 | 800/4000 [02:17<09:08,  5.83it/s, v_num=0]
Validation: |                                             | 0/? [00:00<?, ?it/s]
Validation: |                                             | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|████████████████| 250/250 [00:24<00:00, 10.30it/s]
Epoch 0:  40%|▍| 1600/4000 [04:59<07:29,  5.34it/s, v_num=0, default_validation/
Validation: |                                  

In [ ]:
# factor_importance_rank.csv: ranked key regulators for myoblast with three columns:
#   - factors: regulator names
#   - similarity: cosine similarity of regulator embeddings between highly accessible regions and background region sets
#   - ranks: importance ranking

factor_importance_rank = pd.read_csv("./output_infer_cell_key_regulator/results/factor_importance_rank.csv")
factor_importance_rank.head(n=25)


### Load the fine-tuned checkpoint to infer key regulators for myoblast (skip fine-tuning)

In [12]:
import glob

In [13]:
# if you have already run infer_cell_key_regulator, you can use the fine-tuned checkpoint to infer cell-type-specific key regulators
ft_ckpt_dir = "./output_infer_cell_key_regulator/train/**/*.ckpt"

ft_ckpt = glob.glob(ft_ckpt_dir, recursive=True)[0]

In [14]:
ft_ckpt

'./output_infer_cell_key_regulator/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=1-step=113.ckpt'

In [18]:
# takes approximately 3-5 minutes to run
!chrombert-tools predict_cell_type_master_regulators \
    --cell-type-bw "../data/myoblast_ENCFF149ERN_signal.bigwig" \
    --cell-type-peak "../data/myoblast_ENCFF647RNC_peak.bed" \
    --ft-ckpt {ft_ckpt} \
    --odir "./output_infer_cell_key_regulator_load_cpkt" \
    --genome "hg38" \
    --resolution "1kb"  2> "./tmp/infer_cell_trn.stderr2.log" # redirect stderr to log file

Step 1/3: Building or loading a cell-specific model...
Preparing dataset ...
Region summary - total: 373422, overlapping with ChromBERT: 368260 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 7920
Total regions: 324690
Fast mode: downsampling to 20k regions
Using provided fine-tuned checkpoint: ./output_infer_cell_key_regulator/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=1-step=113.ckpt
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Loading checkpoint from ./output_infer_cell_key_regulator/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=1-step=113.ckpt
Loading from pl module, remove prefix 'model.'
Loading from pl module, replace 'pretrain_model' with 'pretrain_model.chrombert'
Loaded 111/111 parameters
Step 2/3: Preparing region groups fo

In [19]:
factor_importance_rank = pd.read_csv("./output_infer_cell_key_regulator_load_cpkt/results/factor_importance_rank.csv")
factor_importance_rank.head(n=25)

,factors,similarity,rank,embedding_shift
0,myf5,0.077748,1,0.922252
1,yap1,0.098733,2,0.901267
2,myod1,0.112716,3,0.887284
3,tead1,0.116510,4,0.883490
4,cbx6,0.169277,5,0.830723
5,tcf21,0.177595,6,0.822405
6,myog,0.188884,7,0.811116
7,ring1,0.211087,8,0.788913
8,foxo1,0.267976,9,0.732024
9,pax3-foxo1a,0.272871,10,0.727129
